In [1]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
import time

print("Polars version:", pl.__version__)
print("Pandas version:", pd.__version__)

Polars version: 1.44.2
Pandas version: 3.0.6


In [2]:
csv_path = Path("duckdb_100k.csv")

df = pl.read_csv(csv_path)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (100000, 8)
shape: (5, 8)
┌──────────┬────────────┬─────────────┬───────────┬───────────┬──────────┬─────────┬────────┐
│ order_id ┆ order_date ┆ category    ┆ state     ┆ city      ┆ quantity ┆ amount  ┆ profit │
│ ---      ┆ ---        ┆ ---         ┆ ---       ┆ ---       ┆ ---      ┆ ---     ┆ ---    │
│ i64      ┆ str        ┆ str         ┆ str       ┆ str       ┆ i64      ┆ f64     ┆ f64    │
╞══════════╪════════════╪═════════════╪═══════════╪═══════════╪══════════╪═════════╪════════╡
│ 1        ┆ 2018-04-13 ┆ Clothing    ┆ Delhi     ┆ Ahmedabad ┆ 9        ┆ 648.72  ┆ -46.05 │
│ 2        ┆ 2019-03-12 ┆ Clothing    ┆ Rajasthan ┆ Ahmedabad ┆ 12       ┆ 619.08  ┆ -28.12 │
│ 3        ┆ 2018-09-28 ┆ Electronics ┆ Kerala    ┆ Chennai   ┆ 10       ┆ 1076.51 ┆ -4.91  │
│ 4        ┆ 2018-04-17 ┆ Furniture   ┆ Delhi     ┆ Ahmedabad ┆ 13       ┆ 884.38  ┆ 263.14 │
│ 5        ┆ 2018-03-13 ┆ Furniture   ┆ Gujarat   ┆ Jaipur    ┆ 2        ┆ 1997.23 ┆ 82.23  │
└──────────┴───────

In [3]:
print("Rows:", df.height)
print("Columns:", df.width)
print("Missing values:", df.null_count().sum_horizontal().item())
print("Duplicate rows:", df.height - df.unique().height)

Rows: 100000
Columns: 8
Missing values: 0
Duplicate rows: 0


In [4]:
category_sales = (
    df.group_by("category")
    .agg(
        pl.col("amount").sum().alias("total_amount")
    )
    .sort("total_amount", descending=True)
)

category_sales

category,total_amount
str,f64
"""Clothing""",3.5117e7
"""Electronics""",3.4918e7
"""Furniture""",3.4833e7


In [5]:
state_profit = (
    df.group_by("state")
    .agg(
        pl.col("profit").mean().round(2).alias("avg_profit")
    )
    .sort("avg_profit", descending=True)
)

state_profit

state,avg_profit
str,f64
"""Karnataka""",101.01
"""Rajasthan""",100.87
"""Gujarat""",100.63
"""Delhi""",100.13
"""Tamil Nadu""",100.09
"""Kerala""",99.94
"""West Bengal""",99.5
"""Maharashtra""",99.06


In [6]:
city_quantity = (
    df.group_by("city")
    .agg(
        pl.col("quantity").sum().alias("total_quantity")
    )
    .sort("total_quantity", descending=True)
)

city_quantity

city,total_quantity
str,i64
"""Kolkata""",95653
"""Bengaluru""",95249
"""Delhi""",94196
"""Jaipur""",93803
"""Chennai""",93630
"""Kochi""",92596
"""Mumbai""",92248
"""Ahmedabad""",91611


In [7]:
monthly_sales = (
    df.with_columns(
        pl.col("order_date").str.to_date()
    )
    .group_by(
        pl.col("order_date").dt.month().alias("month")
    )
    .agg(
        pl.col("amount").sum().round(2).alias("total_amount")
    )
    .sort("month")
)

monthly_sales

month,total_amount
i8,f64
1,8.9621e6
2,8070841.6
3,8.8080e6
4,8.5481e6
5,8.9029e6
…,…
8,9222672.5
9,8.5711e6
10,8909069.3


In [8]:
filtered_orders = (
    df.filter(pl.col("amount") > 500)
    .group_by("category")
    .agg(
        pl.len().alias("order_count"),
        pl.col("profit").sum().round(2).alias("total_profit")
    )
    .sort("category")
)

filtered_orders

category,order_count,total_profit
str,u32,f64
"""Clothing""",26418,2.6434e6
"""Electronics""",26309,2662335.2
"""Furniture""",26227,2.6327e6


In [9]:
# Eager execution
start = time.perf_counter()

eager_result = (
    df.group_by("category")
    .agg(
        pl.col("amount").sum().alias("total_amount")
    )
    .sort("total_amount", descending=True)
)

eager_time = time.perf_counter() - start

# Lazy execution
start = time.perf_counter()

lazy_result = (
    pl.scan_csv("duckdb_100k.csv")
    .group_by("category")
    .agg(
        pl.col("amount").sum().alias("total_amount")
    )
    .sort("total_amount", descending=True)
    .collect()
)

lazy_time = time.perf_counter() - start

print("Eager execution time:", round(eager_time, 4), "seconds")
print("Lazy execution time:", round(lazy_time, 4), "seconds")

print("\nEager result:")
print(eager_result)

print("\nLazy result:")
print(lazy_result)

print("\nResults match:",
      eager_result.equals(lazy_result))

Eager execution time: 0.0232 seconds
Lazy execution time: 0.0993 seconds

Eager result:
shape: (3, 2)
┌─────────────┬──────────────┐
│ category    ┆ total_amount │
│ ---         ┆ ---          │
│ str         ┆ f64          │
╞═════════════╪══════════════╡
│ Clothing    ┆ 3.5117e7     │
│ Electronics ┆ 3.4918e7     │
│ Furniture   ┆ 3.4833e7     │
└─────────────┴──────────────┘

Lazy result:
shape: (3, 2)
┌─────────────┬──────────────┐
│ category    ┆ total_amount │
│ ---         ┆ ---          │
│ str         ┆ f64          │
╞═════════════╪══════════════╡
│ Clothing    ┆ 3.5117e7     │
│ Electronics ┆ 3.4918e7     │
│ Furniture   ┆ 3.4833e7     │
└─────────────┴──────────────┘

Results match: False


In [10]:
eager_check = (
    eager_result
    .with_columns(pl.col("total_amount").round(2))
    .sort("category")
)

lazy_check = (
    lazy_result
    .with_columns(pl.col("total_amount").round(2))
    .sort("category")
)

print("Eager schema:", eager_check.schema)
print("Lazy schema:", lazy_check.schema)
print("Results match after rounding:", eager_check.equals(lazy_check))

Eager schema: Schema({'category': String, 'total_amount': Float64})
Lazy schema: Schema({'category': String, 'total_amount': Float64})
Results match after rounding: True
